# Chinese Understanding Benchmark v12

Compare three full Transformers models:

1. `google/gemma-4-E4B-it`
2. `Qwen/Qwen3-4B-Instruct-2507`
3. `google/gemma-4-E2B-it`

Based on the working v6 structure, with `raw_normalized` and invalid-output inspection cells added.

## 1. Install dependencies

In [1]:
# Run once, then restart the kernel.
# This notebook does not use torchvision, torchaudio, or MLX.

!pip uninstall -y torchaudio torchvision mlx-vlm || true
!pip install -U transformers datasets accelerate peft trl scikit-learn pandas tqdm sentencepiece


## 2. Configuration

In [6]:
MODELS = {
    "gemma_e4b_it": {"model_id": "google/gemma-4-E4B-it"},
    #"qwen3_4b_instruct_2507": {"model_id": "Qwen/Qwen3-4B-Instruct-2507"},
    #"gemma_e2b_it": {"model_id": "google/gemma-4-E2B-it"},
}

TASKS = ["afqmc", "tnews", "cmnli"]
SPLIT = "validation"
MAX_SAMPLES = 200
DEBUG_N = 3


In [7]:
MODELS

{'gemma_e4b_it': {'model_id': 'google/gemma-4-E4B-it'}}

## 3. Imports and cleanup helpers

In [8]:
import gc
import os
import re
import time
import warnings

import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")


def cleanup_memory():
    gc.collect()
    try:
        torch.mps.empty_cache()
    except Exception:
        pass
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass


def clear_model(model=None, tokenizer=None):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    cleanup_memory()


## 4. Label mappings

In [9]:
TASK_SPECS = {
    "afqmc": {
        "label_id_to_name": {"0": "不同", "1": "相同"},
        "label_name_to_id": {"不同": "0", "相同": "1"},
        "aliases": {
            "不同": ["不同", "不相同", "不一致", "不等价", "语义不同", "否", "no", "false", "0"],
            "相同": ["相同", "一致", "等价", "同义", "语义相同", "是", "yes", "true", "1"],
        },
        "label_id_style": "fixed_manual",
    },
    "cmnli": {
        # Observed HF/CLUE mapping: 0 = neutral, 1 = entailment, 2 = contradiction
        "label_id_to_name": {"0": "中立", "1": "蕴含", "2": "矛盾"},
        "label_name_to_id": {"中立": "0", "蕴含": "1", "矛盾": "2"},
        "aliases": {
            "中立": ["中立", "无法判断", "不能确定", "无关", "neutral", "0"],
            "蕴含": ["蕴含", "包含", "推出", "可以推出", "entailment", "entails", "1"],
            "矛盾": ["矛盾", "冲突", "contradiction", "contradict", "2"],
        },
        "label_id_style": "fixed_manual",
    },
    "tnews": {
        # Correct CLUE TNEWS mapping used in your working v6-style runs.
        "label_id_to_name": {
            "0": "故事", "1": "文化", "2": "娱乐", "3": "体育", "4": "财经",
            "5": "房产", "6": "汽车", "7": "教育", "8": "科技", "9": "国际",
            "10": "旅游", "11": "军事", "12": "股票", "13": "农业", "14": "电竞",
        },
        "label_name_to_id": {
            "故事": "0", "文化": "1", "娱乐": "2", "体育": "3", "财经": "4",
            "房产": "5", "汽车": "6", "教育": "7", "科技": "8", "国际": "9",
            "旅游": "10", "军事": "11", "股票": "12", "农业": "13", "电竞": "14",
        },
        "aliases": {
            "故事": ["故事"], "文化": ["文化"], "娱乐": ["娱乐"], "体育": ["体育"], "财经": ["财经"],
            "房产": ["房产"], "汽车": ["汽车"], "教育": ["教育"], "科技": ["科技"], "国际": ["国际"],
            "旅游": ["旅游"], "军事": ["军事"], "股票": ["股票"], "农业": ["农业"], "电竞": ["电竞"],
        },
        "label_id_style": "fixed_manual_tnews",
    },
}


def get_task_spec(task):
    return TASK_SPECS[task]


def label_id_to_name(label_id, spec):
    return spec["label_id_to_name"].get(str(label_id), str(label_id))


def pred_id_to_name(pred_id, spec):
    if pred_id == "__invalid__":
        return "__invalid__"
    return spec["label_id_to_name"].get(str(pred_id), "__invalid__")


## 5. Prompts and parser

In [10]:
def build_prompt(task, ex, spec):
    labels = "、".join(spec["label_name_to_id"].keys())

    if task == "afqmc":
        return f"""你是中文二分类器。只输出“相同”或“不同”其中一个词，不要解释。

句子1：{ex["sentence1"]}
句子2：{ex["sentence2"]}

语义是否相同？答案："""

    if task == "cmnli":
        return f"""你是中文自然语言推理分类器。只输出“蕴含”、“中立”或“矛盾”其中一个词，不要解释。

前提：{ex["sentence1"]}
假设：{ex["sentence2"]}

关系是？答案："""

    if task == "tnews":
        return f"""你是中文新闻标题分类器。只能从以下类别中选择一个输出，不要解释。

可选类别：{labels}

标题：{ex["sentence"]}

类别："""

    raise ValueError(f"Unsupported task: {task}")


def normalize_output(text):
    text = str(text).strip().lower()
    artifacts = ["<bos>", "<eos>", "<pad>", "<start_of_turn>", "<end_of_turn>", "model", "assistant", "user"]
    for token in artifacts:
        text = text.replace(token, " ")
    text = text.replace("答案：", " ").replace("答案:", " ")
    text = text.replace("类别：", " ").replace("类别:", " ")
    text = text.replace("标签：", " ").replace("标签:", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"[。，“”，、；;:：\[\]\(\)（）\"'`]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_pred_id(task, raw_output, spec):
    text_norm = normalize_output(raw_output)

    # Exact normalized alias match first.
    for canonical_name, aliases in spec["aliases"].items():
        for alias in aliases:
            alias_norm = normalize_output(alias)
            if text_norm == alias_norm:
                return spec["label_name_to_id"].get(canonical_name, "__invalid__")

    # Substring match second.
    alias_items = []
    for canonical_name, aliases in spec["aliases"].items():
        for alias in aliases:
            alias_items.append((canonical_name, normalize_output(alias)))
    alias_items = sorted(alias_items, key=lambda x: len(x[1]), reverse=True)

    for canonical_name, alias_norm in alias_items:
        if alias_norm and alias_norm in text_norm:
            return spec["label_name_to_id"].get(canonical_name, "__invalid__")

    return "__invalid__"


## 6. Model loading and generation

In [11]:
def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    return tokenizer, model


@torch.no_grad()
def generate_answer(tokenizer, model, prompt, max_new_tokens=8):
    inputs = tokenizer(prompt, return_tensors="pt")
    try:
        inputs = inputs.to(model.device)
    except Exception:
        pass
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return decoded.strip()


## 7. Evaluation loop

In [12]:
def evaluate_loaded_model(model_key, model_id, tokenizer, model):
    model_summaries = []
    model_rows = []

    for task in TASKS:
        dataset = load_dataset("clue", task, split=SPLIT)
        dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
        spec = get_task_spec(task)

        y_true = []
        y_pred = []
        start = time.time()

        for idx, ex in enumerate(tqdm(dataset, desc=f"{model_key}/{task}")):
            prompt = build_prompt(task, ex, spec)
            raw = generate_answer(tokenizer, model, prompt, max_new_tokens=8)

            gold_id = str(ex["label"])
            gold_name = label_id_to_name(gold_id, spec)
            pred_id = extract_pred_id(task, raw, spec)
            pred_name = pred_id_to_name(pred_id, spec)

            y_true.append(gold_id)
            y_pred.append(pred_id)

            row = {
                "model_key": model_key,
                "model_id": model_id,
                "task": task,
                "split": SPLIT,
                "idx": idx,
                "gold_id": gold_id,
                "gold_name": gold_name,
                "raw": repr(raw),
                "raw_normalized": normalize_output(raw),
                "pred_name": pred_name,
                "pred_id": pred_id,
            }
            model_rows.append(row)

            if idx < DEBUG_N:
                print(row)

        elapsed = time.time() - start
        invalid_count = sum(p == "__invalid__" for p in y_pred)
        y_pred_for_score = [p if p != "__invalid__" else "-1" for p in y_pred]

        summary = {
            "model_key": model_key,
            "model_id": model_id,
            "task": task,
            "split": SPLIT,
            "samples": len(y_true),
            "accuracy": accuracy_score(y_true, y_pred_for_score),
            "macro_f1": f1_score(y_true, y_pred_for_score, average="macro", zero_division=0),
            "invalid_rate": invalid_count / len(y_pred),
            "invalid_count": invalid_count,
            "seconds": elapsed,
            "samples_per_second": len(y_true) / elapsed if elapsed > 0 else None,
            "label_id_style": spec["label_id_style"],
        }
        print(summary)
        model_summaries.append(summary)

    return model_summaries, model_rows


def evaluate_one_model(model_key, model_cfg):
    model_id = model_cfg["model_id"]
    cleanup_memory()
    print(f"\n=== Loading {model_key}: {model_id} ===")
    tokenizer = None
    model = None
    try:
        tokenizer, model = load_model(model_id)
        summaries, rows = evaluate_loaded_model(model_key, model_id, tokenizer, model)
    finally:
        print(f"Clearing model from memory: {model_key}")
        clear_model(model, tokenizer)
    return summaries, rows


## 8. Run benchmark

In [13]:
all_summaries = []
all_rows = []

for model_key, model_cfg in MODELS.items():
    summaries, rows = evaluate_one_model(model_key, model_cfg)
    all_summaries.extend(summaries)
    all_rows.extend(rows)

summary_df = pd.DataFrame(all_summaries)
detail_df = pd.DataFrame(all_rows)
summary_df



=== Loading gemma_e4b_it: google/gemma-4-E4B-it ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

gemma_e4b_it/afqmc:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '不同', 'raw': "'：：：\\n\\n语义是否相同？'", 'raw_normalized': '语义是否相同？', 'pred_name': '相同', 'pred_id': '1'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'idx': 1, 'gold_id': '0', 'gold_name': '不同', 'raw': "'：\\n\\n句子是否相同？答案：'", 'raw_normalized': '句子是否相同？', 'pred_name': '相同', 'pred_id': '1'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'idx': 2, 'gold_id': '1', 'gold_name': '相同', 'raw': "'：：：\\n\\n语义是否相同？'", 'raw_normalized': '语义是否相同？', 'pred_name': '相同', 'pred_id': '1'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'samples': 200, 'accuracy': 0.35, 'macro_f1': 0.19081406881818874, 'invalid_rate': 0.035, 'invalid_count': 7, 'seconds': 208.9256911277771, 'samples_per_seco

gemma_e4b_it/tnews:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'tnews', 'split': 'validation', 'idx': 0, 'gold_id': '2', 'gold_name': '娱乐', 'raw': "'故事、文化、娱乐、体育、'", 'raw_normalized': '故事 文化 娱乐 体育', 'pred_name': '故事', 'pred_id': '0'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'tnews', 'split': 'validation', 'idx': 1, 'gold_id': '9', 'gold_name': '国际', 'raw': "'故事、文化、娱乐、体育、'", 'raw_normalized': '故事 文化 娱乐 体育', 'pred_name': '故事', 'pred_id': '0'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'tnews', 'split': 'validation', 'idx': 2, 'gold_id': '4', 'gold_name': '财经', 'raw': "'::::::::'", 'raw_normalized': '', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'tnews', 'split': 'validation', 'samples': 200, 'accuracy': 0.02, 'macro_f1': 0.010800438596491227, 'invalid_rate': 0.255, 'invalid_count': 51, 'seconds': 229.97087001800537, 'samples_pe

gemma_e4b_it/cmnli:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'cmnli', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '中立', 'raw': "'前提：新的权利已经足够好了'", 'raw_normalized': '前提 新的权利已经足够好了', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'cmnli', 'split': 'validation', 'idx': 1, 'gold_id': '1', 'gold_name': '蕴含', 'raw': "'嗯，我不知道，我对他有'", 'raw_normalized': '嗯 我不知道 我对他有', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'cmnli', 'split': 'validation', 'idx': 2, 'gold_id': '2', 'gold_name': '矛盾', 'raw': "'：：：：：：：：'", 'raw_normalized': '', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'cmnli', 'split': 'validation', 'samples': 200, 'accuracy': 0.01, 'macro_f1': 0.01098901098901099, 'invalid_rate': 0.97, 'invalid_count': 194, 'se

,model_key,model_id,task,split,samples,accuracy,macro_f1,invalid_rate,invalid_count,seconds,samples_per_second,label_id_style
0,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,200,0.35,0.190814,0.035,7,208.925691,0.957278,fixed_manual
1,gemma_e4b_it,google/gemma-4-E4B-it,tnews,validation,200,0.02,0.010800,0.255,51,229.970870,0.869675,fixed_manual_tnews
2,gemma_e4b_it,google/gemma-4-E4B-it,cmnli,validation,200,0.01,0.010989,0.970,194,215.983982,0.925995,fixed_manual


## 9. Inspect details

In [14]:
detail_df.head(20)

,model_key,model_id,task,split,idx,gold_id,gold_name,raw,raw_normalized,pred_name,pred_id
0,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,0,0,不同,'：：：\n\n语义是否相同？',语义是否相同？,相同,1
1,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,1,0,不同,'：\n\n句子是否相同？答案：',句子是否相同？,相同,1
2,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,2,1,相同,'：：：\n\n语义是否相同？',语义是否相同？,相同,1
3,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,3,0,不同,'：：：\n\n句子1：为什么',句子1 为什么,相同,1
4,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,4,0,不同,'：：：\n\n语义是否相同？',语义是否相同？,相同,1
5,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,5,1,相同,'：：：\n\n语义是否相同？',语义是否相同？,相同,1
6,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,6,0,不同,'：：\n\n语义是否相同？答案',语义是否相同？答案,相同,1
7,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,7,0,不同,'：：：\n\n语义是否相同？',语义是否相同？,相同,1
8,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,8,0,不同,'：：：\n\n语义是否相同？',语义是否相同？,相同,1
9,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,9,0,不同,'：：：\n\n语义是否相同？',语义是否相同？,相同,1


In [15]:
invalid_df = detail_df[detail_df["pred_id"] == "__invalid__"].copy()
print("Invalid count:", len(invalid_df))
invalid_df.head(50)


Invalid count: 252


,model_key,model_id,task,split,idx,gold_id,gold_name,raw,raw_normalized,pred_name,pred_id
57,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,57,0,不同,'：\n\n---',---,__invalid__,__invalid__
92,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,92,0,不同,'：：：\n\n句子2：借',句子2 借,__invalid__,__invalid__
112,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,112,0,不同,'：：：：：：：：',,__invalid__,__invalid__
115,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,115,0,不同,'：',,__invalid__,__invalid__
142,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,142,1,相同,'：：：：：：：：',,__invalid__,__invalid__
191,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,191,1,相同,'语义：怎么无法开通蚂蚁',语义 怎么无法开通蚂蚁,__invalid__,__invalid__
192,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,192,0,不同,'：',,__invalid__,__invalid__
202,gemma_e4b_it,google/gemma-4-E4B-it,tnews,validation,2,4,财经,'::::::::',,__invalid__,__invalid__
204,gemma_e4b_it,google/gemma-4-E4B-it,tnews,validation,4,10,旅游,'房客、不素质\n\n标题：',房客 不素质 标题,__invalid__,__invalid__
208,gemma_e4b_it,google/gemma-4-E4B-it,tnews,validation,8,5,房产,'：：：：：：：：',,__invalid__,__invalid__


In [16]:
# Inspect Gemma E4B-it invalid CMNLI outputs
detail_df[
    (detail_df["model_key"] == "gemma_e4b_it") &
    (detail_df["task"] == "cmnli") &
    (detail_df["pred_id"] == "__invalid__")
][["idx", "gold_id", "gold_name", "raw", "raw_normalized", "pred_name", "pred_id"]].head(50)


,idx,gold_id,gold_name,raw,raw_normalized,pred_name,pred_id
400,0,0,中立,'前提：新的权利已经足够好了',前提 新的权利已经足够好了,__invalid__,__invalid__
401,1,1,蕴含,'嗯，我不知道，我对他有',嗯 我不知道 我对他有,__invalid__,__invalid__
402,2,2,矛盾,'：：：：：：：：',,__invalid__,__invalid__
403,3,2,矛盾,'：\n\n关系是？答案：：',关系是？,__invalid__,__invalid__
404,4,2,矛盾,'：：：\n假设：我们在垃圾',假设 我们在垃圾,__invalid__,__invalid__
405,5,2,矛盾,'```\n```\n```\n```',,__invalid__,__invalid__
406,6,0,中立,'森先生的大部分作品都可以在欧洲',森先生的大部分作品都可以在欧洲,__invalid__,__invalid__
407,7,2,矛盾,'：：：。\n前提：如果',前提 如果,__invalid__,__invalid__
408,8,0,中立,'关系是？关系是？',关系是？关系是？,__invalid__,__invalid__
409,9,2,矛盾,'',,__invalid__,__invalid__


In [17]:
# Inspect Gemma E4B-it all CMNLI outputs
detail_df[
    (detail_df["model_key"] == "gemma_e4b_it") &
    (detail_df["task"] == "cmnli")
][["idx", "gold_id", "gold_name", "raw", "raw_normalized", "pred_name", "pred_id"]].head(50)


,idx,gold_id,gold_name,raw,raw_normalized,pred_name,pred_id
400,0,0,中立,'前提：新的权利已经足够好了',前提 新的权利已经足够好了,__invalid__,__invalid__
401,1,1,蕴含,'嗯，我不知道，我对他有',嗯 我不知道 我对他有,__invalid__,__invalid__
402,2,2,矛盾,'：：：：：：：：',,__invalid__,__invalid__
403,3,2,矛盾,'：\n\n关系是？答案：：',关系是？,__invalid__,__invalid__
404,4,2,矛盾,'：：：\n假设：我们在垃圾',假设 我们在垃圾,__invalid__,__invalid__
405,5,2,矛盾,'```\n```\n```\n```',,__invalid__,__invalid__
406,6,0,中立,'森先生的大部分作品都可以在欧洲',森先生的大部分作品都可以在欧洲,__invalid__,__invalid__
407,7,2,矛盾,'：：：。\n前提：如果',前提 如果,__invalid__,__invalid__
408,8,0,中立,'关系是？关系是？',关系是？关系是？,__invalid__,__invalid__
409,9,2,矛盾,'',,__invalid__,__invalid__


In [18]:
# Invalid count by model/task
detail_df.assign(is_invalid=detail_df["pred_id"].eq("__invalid__")).groupby(
    ["model_key", "task"]
)["is_invalid"].agg(["sum", "mean"])


sum   mean
model_key    task             
gemma_e4b_it afqmc    7  0.035
             cmnli  194  0.970
             tnews   51  0.255

## 10. Save results

In [ ]:
os.makedirs("results", exist_ok=True)
summary_path = "results/chinese_understanding_summary_v12.csv"
detail_path = "results/chinese_understanding_details_v12.csv"
summary_df.to_csv(summary_path, index=False)
detail_df.to_csv(detail_path, index=False)
print("Saved:")
print(summary_path)
print(detail_path)


## 11. Optional LoRA fine-tuning scaffold

In [ ]:
RUN_LORA_TRAINING = False

if RUN_LORA_TRAINING:
    from peft import LoraConfig
    from trl import SFTTrainer
    from transformers import TrainingArguments

    def format_sft_example(task, ex, spec):
        prompt = build_prompt(task, ex, spec)
        answer = label_id_to_name(ex["label"], spec)
        return prompt + answer

    def train_lora(model_id, task="afqmc", output_dir="adapters/lora_model", max_train_samples=1000):
        dataset = load_dataset("clue", task, split="train")
        dataset = dataset.select(range(min(max_train_samples, len(dataset))))
        spec = get_task_spec(task)
        dataset = dataset.map(lambda ex: {"text": format_sft_example(task, ex, spec)})
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            task_type="CAUSAL_LM",
        )
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            num_train_epochs=1,
            logging_steps=20,
            save_steps=500,
            fp16=True,
            report_to="none",
        )
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=dataset,
            dataset_text_field="text",
            peft_config=lora_config,
            args=training_args,
            max_seq_length=512,
        )
        trainer.train()
        trainer.save_model(output_dir)
        clear_model(model, tokenizer)

    # Example:
    # train_lora("Qwen/Qwen3-4B-Instruct-2507", task="afqmc", output_dir="adapters/qwen3_4b_afqmc")
